In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import shutil
import os

# 1. 압축 파일 경로 (구글 드라이브 내 위치)
zip_file_path = '/content/drive/MyDrive/project_v5.zip'

# 2. 압축을 풀 폴더 경로
extract_to_path = '/content/dataset/'

# 폴더가 없으면 자동으로 생성
if not os.path.exists(extract_to_path):
    os.makedirs(extract_to_path)

# 3. 압축 해제 실행
try:
    print("압축 해제 중...")
    shutil.unpack_archive(zip_file_path, extract_to_path)
    print(f"압축 해제 완료! 저장 경로: {extract_to_path}")
except Exception as e:
    print(f"압축 해제 실패: {e}")

⏳ 압축 해제 중... 잠시만 기다려 주세요.
✅ 압축 해제 완료! 저장 경로: /content/dataset/


In [ ]:
import os
import glob

# 하위 폴더까지 모두 검색하기 위해 recursive=True와 ** 사용
train_labels = glob.glob('/content/dataset/dataset/labels/train/**/*.txt', recursive=True)
valid_labels = glob.glob('/content/dataset/dataset/labels/valid/**/*.txt', recursive=True)
all_labels = train_labels + valid_labels

class_counts = {0: 0, 1: 0}
total_boxes = 0

print(f"총 {len(all_labels)}개의 라벨 파일 분석 시작...")

for label_path in all_labels:
    try:
        with open(label_path, 'r') as f:
            for line in f:
                parts = line.strip().split()
                if not parts: continue

                class_id = int(parts[0])
                if class_id in class_counts:
                    class_counts[class_id] += 1
                    total_boxes += 1
    except Exception as e:
        print(f"Error reading {label_path}: {e}")

print("=" * 50)
print(f"[라벨 전수조사 결과] 총 라벨 파일 수: {len(all_labels)}개")
print(f"전체 Bounding Box 개수: {total_boxes}개")
print("-" * 50)
print(f"Class 0 (YES_Helmet) 개수 : {class_counts[0]:,}개")
print(f"Class 1 (NO_Helmet) 개수   : {class_counts[1]:,}개")
print("=" * 50)

# 추가 팁: 클래스 분포 비율 확인
if total_boxes > 0:
    ratio0 = (class_counts[0] / total_boxes) * 100
    ratio1 = (class_counts[1] / total_boxes) * 100
    print(f"분포 비율 -> YES: {ratio0:.1f}%, NO: {ratio1:.1f}%")

🚀 총 4000개의 라벨 파일 분석 시작...
📈 [라벨 전수조사 결과] 총 라벨 파일 수: 4000개
📦 전체 Bounding Box 개수: 19840개
--------------------------------------------------
🪖 Class 0 (YES_Helmet) 개수 : 15,232개
🧑 Class 1 (NO_Helmet) 개수   : 4,608개
분포 비율 -> YES: 76.8%, NO: 23.2%


In [ ]:
!pip install ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 kB 1.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 12.4 MB/s eta 0:00:00


In [ ]:
from ultralytics import YOLO

model = YOLO('best.pt')

model.train(
    data='/content/dataset/data.yaml',
    optimizer='AdamW',
    epochs=40,             # 많은 에폭보다는 짧고 굵게 학습
    imgsz=640,
    lr0=0.005,             # 0.0001보다 훨씬 높은 학습률로 강제 교정
    lrf=0.001,              # 학습 후반에 lr이 급격히 줄어들게 하여 안정화
    momentum=0.937,        # 기존 모멘텀 유지
    weight_decay=0.0005,   # 과적합 방지
    freeze=0,              # 전체 레이어 학습 (강제 수정)
    cls=2.0,  # 클래스 분류 손실 가중치 증가 (NO_Helmet 인식률 향상에 도움)
    # class_weights를 조절하여 NO_Helmet을 더 잘 찾게 함
    # 혹은 loss 함수 가중치를 직접 조절
)

Ultralytics 8.4.58 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA L4, 22563MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=2.0, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=40, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=0, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.005, lrf=0.001, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=best.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train-2, nbs=64, nms=False, opset=None, optimize=False, optimizer=AdamW, overlap_mask=True, patience=100, pe

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x791aa72e5c40>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.04804

In [ ]:
from ultralytics import YOLO
import cv2
model = YOLO('/content/combination_yolo26s.pt')

# 영상 경로 설정
video_path = '/content/test.mp4'

# 추론 실행 및 결과 저장
# conf=0.3: 신뢰도 임계값, 원하시는 수치로 조정 가능합니다.
# save=True: 결과를 영상으로 저장합니다.
results = model.predict(
    source=video_path,
    conf=0.3,
    save=True,
    imgsz=640,
    #device=0
    stream=True         # 대용량 영상 처리를 위한 스트리밍 방식
)

# 4. 결과 출력
# stream=True를 사용하면 generator 객체가 반환되므로 루프를 돌려야 합니다.
for r in results:
    # 각 프레임별로 추론 결과를 처리하거나 시각화할 수 있습니다.
    # r.plot()을 사용하면 바운딩 박스가 그려진 이미지가 생성됩니다.
    annotated_frame = r.plot()

    # 화면 표시를 원하시면 cv2.imshow를 쓰지만, 코랩에서는
    # 자동으로 save=True 설정에 의해 파일로 저장됩니다.
    pass

print("영상 추론 및 결과 저장 완료!")


video 1/1 (frame 1/102) /content/test.mp4: 384x640 1 person, 540.0ms
video 1/1 (frame 2/102) /content/test.mp4: 384x640 1 helmet, 1 person, 488.6ms
video 1/1 (frame 3/102) /content/test.mp4: 384x640 1 helmet, 1 person, 393.8ms
video 1/1 (frame 4/102) /content/test.mp4: 384x640 1 person, 312.5ms
video 1/1 (frame 5/102) /content/test.mp4: 384x640 2 helmets, 1 person, 320.8ms
video 1/1 (frame 6/102) /content/test.mp4: 384x640 1 helmet, 1 person, 328.6ms
video 1/1 (frame 7/102) /content/test.mp4: 384x640 2 persons, 331.1ms
video 1/1 (frame 8/102) /content/test.mp4: 384x640 1 person, 345.1ms
video 1/1 (frame 9/102) /content/test.mp4: 384x640 3 persons, 313.0ms
video 1/1 (frame 10/102) /content/test.mp4: 384x640 2 persons, 316.1ms
video 1/1 (frame 11/102) /content/test.mp4: 384x640 1 person, 334.2ms
video 1/1 (frame 12/102) /content/test.mp4: 384x640 2 persons, 313.0ms
video 1/1 (frame 13/102) /content/test.mp4: 384x640 2 persons, 329.0ms
video 1/1 (frame 14/102) /content/test.mp4: 384x640 